<a href="https://colab.research.google.com/github/PhcPh4m/ETL-Stream2DuckDB/blob/main/notebooks/02_batch_etl_api_to_duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
#Clone repo & install requirements.txt
!git clone https://github.com/PhcPh4m/ETL-Stream2DuckDB.git
%cd ETL-Stream2DuckDB
!pip install -r requirements.txt


Cloning into 'ETL-Stream2DuckDB'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 18 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 10.04 KiB | 10.04 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/ETL-Stream2DuckDB/ETL-Stream2DuckDB/ETL-Stream2DuckDB/ETL-Stream2DuckDB


In [17]:
#Set API key
import os
import getpass
os.environ['OPENWEATHER_API_KEY'] = getpass.getpass('OpenWeather API Key:')

OpenWeather API Key:··········


In [16]:
import os, requests

key = os.getenv("OPENWEATHER_API_KEY")
print("Key set:", bool(key))
if key:
    k = key.strip()
    print("Length:", len(k))
    # show masked key: first 4 and last 4 chars
    print("Masked key:", f"{k[:4]}...{k[-4:]}")
else:
    print("OPENWEATHER_API_KEY is empty. Set it with getpass or os.environ.")

# Try a direct request to see API response body
params = {"q": "Ho Chi Minh", "appid": key.strip() if key else "", "units": "metric"}
r = requests.get("https://api.openweathermap.org/data/2.5/weather", params=params, timeout=10)
print("status:", r.status_code)
print("body:", r.text)


Key set: True
Length: 32
Masked key: bb6e...705f
status: 401
body: {"cod":401, "message": "Invalid API key. Please see https://openweathermap.org/faq#error401 for more info."}


In [19]:
#run ingest
from src.etl_api import fetch_weather, save_raw_jsonl
import os
from datetime import datetime

api_key = os.getenv("OPENWEATHER_API_KEY")
if not api_key:
    raise SystemExit("OPENWEATHER_API_KEY not set")

city = "Ho Chi Minh"
rec = fetch_weather(city, api_key)
rec["ingest_ts"] = datetime.utcnow().isoformat()
out_path = f"data/raw/weather_{datetime.utcnow().strftime('%Y%m%d')}.jsonl"
save_raw_jsonl(rec, out_path)
print("Saved to", out_path)


Saved to data/raw/weather_20260224.jsonl


/tmp/ipython-input-950666737.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec["ingest_ts"] = datetime.utcnow().isoformat()
/tmp/ipython-input-950666737.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  out_path = f"data/raw/weather_{datetime.utcnow().strftime('%Y%m%d')}.jsonl"
